# **Exception Handling — Coding Exercise**

**Learning Objective**

This exercise practices **raising** and **handling** exceptions, designing a **custom exception hierarchy**, and writing **defensive** classes that never end up in an invalid state.

## **Laboratory Exercise: Campus Wallet Accounts**

### **Scenario**

Students load money into a **campus wallet** to pay at the canteen and bookstore. Each account has an **owner** and a **balance**. Money can be **deposited**, **withdrawn**, and **transferred** between accounts. The program must reject invalid operations with clear messages — and must never lose or create money by accident.

```{mermaid}
classDiagram
Exception <|-- WalletError
WalletError <|-- InvalidAmountError
WalletError <|-- InsufficientFundsError
WalletError <|-- AccountFrozenError
class InsufficientFundsError {
    +float balance
    +float requested
    +shortfall() float
}
```

### **Task 1 — Custom Exceptions**

Create the exception hierarchy shown above:

* `WalletError` — the base class for all wallet errors;
* `InvalidAmountError` — the amount is zero, negative, or not a number;
* `InsufficientFundsError` — stores `balance` and `requested` as attributes, has a `shortfall()` method, and its message is `Insufficient funds: balance 120.00, requested 500.00`;
* `AccountFrozenError` — the account is frozen.

```python
error = InsufficientFundsError(120, 500)
print(error)
print(error.shortfall())
print(isinstance(error, WalletError), isinstance(error, Exception))
```

Expected output:

```text
Insufficient funds: balance 120.00, requested 500.00
380
True True
```

### **Task 2 — A Defensive `Account` Class**

Create `Account(owner, balance=0)` with:

* a **read-only** `balance` property;
* `deposit(amount)` and `withdraw(amount)` that raise `InvalidAmountError` for non-numeric, zero, or negative amounts, and `InsufficientFundsError` when withdrawing too much;
* `freeze()`, after which **every** deposit or withdrawal raises `AccountFrozenError`;
* the constructor raises `InvalidAmountError` if the initial balance is negative.

Use a **private helper** method `_check_amount(amount)` so the validation is written only once.

```python
acct = Account("Ana", 1000)
acct.withdraw(250)
acct.deposit(100)
print(acct.balance)

for action in (lambda: acct.withdraw(5000),
               lambda: acct.deposit(-20),
               lambda: acct.deposit("50"),
               lambda: Account("Ben", -1)):
    try:
        action()
    except WalletError as error:
        print(f"{type(error).__name__}: {error}")
```

Expected output:

```text
850
InsufficientFundsError: Insufficient funds: balance 850.00, requested 5000.00
InvalidAmountError: Amount must be a positive number, got -20
InvalidAmountError: Amount must be a positive number, got '50'
InvalidAmountError: Initial balance cannot be negative.
```

### **Task 3 — `try`, `except`, `else`, `finally`**

Write a function `pay(account, amount, item)` that:

* attempts the withdrawal;
* on `InsufficientFundsError`, returns `"Declined: need ₱X.XX more for <item>"` using `shortfall()`;
* on any other `WalletError`, returns `"Declined: <message>"`;
* in the `else` block, returns `"Paid ₱X.XX for <item>"`;
* in the `finally` block, prints `"[log] <owner> attempted <item>"`.

```python
acct = Account("Carlo", 200)
print(pay(acct, 150, "lunch"))
print(pay(acct, 120, "notebook"))
acct.freeze()
print(pay(acct, 10, "candy"))
```

Expected output:

```text
[log] Carlo attempted lunch
Paid ₱150.00 for lunch
[log] Carlo attempted notebook
Declined: need ₱70.00 more for notebook
[log] Carlo attempted candy
Declined: Account of Carlo is frozen.
```

> Why does the `[log]` line appear **before** the returned message, even though `finally` comes last in the code?

### **Task 4 — Safe Transfers**

Write a function `transfer(source, target, amount)` that withdraws from `source` and deposits into `target`.

If the **deposit** fails (for example, the target is frozen), the money must be **returned** to the source before the exception continues upward. Use `raise WalletError("Transfer failed") from error` so the original cause is kept.

```python
a = Account("Ana", 500)
b = Account("Ben", 100)
transfer(a, b, 200)
print(a.balance, b.balance)

b.freeze()
try:
    transfer(a, b, 100)
except WalletError as error:
    print(error, "| cause:", type(error.__cause__).__name__)
print(a.balance, b.balance)
```

Expected output:

```text
300 300
Transfer failed | cause: AccountFrozenError
300 300
```

### **Task 5 — Reflection**

1. Why is it useful for all custom exceptions to inherit from `WalletError`?
2. Your `Account` methods `raise` exceptions but never `print()` them. Where should printing happen, and why?
3. What could go wrong in Task 4 without the rollback?

## **Submission Requirements**

Submit **one Jupyter Notebook** containing all tasks in order. Each task should have:

* the code that solves it;
* the output produced by running the code; and
* the Markdown answers to the questions asked.

Make sure the notebook runs from top to bottom without errors (**Kernel → Restart & Run All**) before submitting.

## **References**

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Python Software Foundation. (2026). *Built-in exceptions*. https://docs.python.org/3/library/exceptions.html

Python Software Foundation. (2026). *The Python tutorial: 8. Errors and exceptions*. https://docs.python.org/3/tutorial/errors.html